In [40]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

project_root = Path.cwd().resolve().parent.parent
processed_dir = project_root / 'src' / 'data' / 'processed'

# Importing the PitStrategyNet model
from src.models.model import PitStrategyNet

# Load data
X_train = np.load(processed_dir / 'X_train.npy')
y_train = np.load(processed_dir / 'y_train.npy')

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

X_train: (13062, 23), y_train: (13062, 1)


In [41]:
class LapDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(LapDataset(X_train, y_train), batch_size=32, shuffle=True)

In [42]:
input_dim = X_train.shape[1]
model = PitStrategyNet(input_dim)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

pit_count = (y_train == 1).sum()
non_pit_count = (y_train == 0).sum()
pos_weight = torch.tensor([non_pit_count / pit_count], dtype=torch.float32)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Input dim:  {input_dim}")
print(f"pos_weight: {pos_weight.item():.2f}")

Input dim:  23
pos_weight: 29.66


In [43]:
num_epochs = 150

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.9

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {train_loss/len(train_loader):.4f}")

# Save weights
models_dir = project_root / 'src' / 'models'
models_dir.mkdir(exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': input_dim,
}, models_dir / 'pit_strategy_net.pth')

print("Model saved.")


Epoch 10/150 | Train Loss: 0.0599
Epoch 20/150 | Train Loss: 0.0260
Epoch 30/150 | Train Loss: 0.0575
Epoch 40/150 | Train Loss: 0.0852
Epoch 50/150 | Train Loss: 0.0147
Epoch 60/150 | Train Loss: 0.0341
Epoch 70/150 | Train Loss: 0.0464
Epoch 80/150 | Train Loss: 0.0329
Epoch 90/150 | Train Loss: 0.0504
Epoch 100/150 | Train Loss: 0.0164
Epoch 110/150 | Train Loss: 0.0816
Epoch 120/150 | Train Loss: 0.0554
Epoch 130/150 | Train Loss: 0.0204
Epoch 140/150 | Train Loss: 0.2634
Epoch 150/150 | Train Loss: 0.0468
Model saved.
